In [1]:
import pandas as pd
import numpy as np
import sys
import os


In [2]:
sys.path.append(os.path.abspath(".."))

In [3]:
from src.data_utils import (load_data)
from src.cleaning_utils import (drop_irrelevant_columns,
                                standardize_placeholders,
                                to_category,
                                handle_missing_values,
                                log_transform_skewed_columns,
                                cap_outliers_iqr,
                                drop_duplicates,
                                group_rare_categories,
                                convert_to_datetime,
                                save_cleaned_data
                                )



In [4]:
X_train = load_data("../data/X_train.csv")
y_train = load_data("../data/y_train.csv")

df_raw = X_train.merge(y_train, on="id")

DataFrame successfully loaded.
Shape: (59400, 40)
DataFrame successfully loaded.
Shape: (59400, 2)


In [5]:
cols_to_drop = [
    "id",                # unique identifier
    "wpt_name",          # too many unique values
    "num_private",       # mostly zeros / unclear meaning
    "recorded_by",        # constant value
    "scheme_name",       # too many unique values
    "extraction_type_group",  # already captured by extraction_type
    "payment_type",     # already captured by payment
    "quantity_group",   # already captured by quantity
    "waterpoint_type_group",  # already captured by waterpoint_type
    "subvillage"       # too many unique values
]

In [6]:
df_clean = drop_irrelevant_columns(df_raw, cols_to_drop)

Dropped columns: ['id', 'wpt_name', 'num_private', 'subvillage', 'recorded_by', 'scheme_name', 'extraction_type_group', 'payment_type', 'quantity_group', 'waterpoint_type_group']


In [7]:
df_clean = standardize_placeholders(df_clean)

 Placeholder values replaced:
{'funder': 781, 'installer': 781, 'management': 561, 'management_group': 561, 'payment': 8157, 'water_quality': 1876, 'quality_group': 1876, 'quantity': 789, 'source': 66, 'source_class': 278}


In [8]:
df_clean = to_category(df_clean, cols=["district_code", "region_code"])

Converted to category: ['region_code', 'district_code']


In [10]:
df_clean = convert_to_datetime(df_clean, cols=["date_recorded", "construction_year"])

Converted to datetime: ['date_recorded', 'construction_year']


In [11]:
df_clean = drop_duplicates(df_clean)

Dropped 534 duplicate rows


In [12]:
handle_missing_values(df_clean, strategy="fill", fill_value="Unknown")

Filled missing values (excluding: [])


,amount_tsh,date_recorded,funder,gps_height,installer,longitude,latitude,basin,region,region_code,...,management_group,payment,water_quality,quality_group,quantity,source,source_type,source_class,waterpoint_type,status_group
0,6000.0,2011-03-14,Roman,1390,Roman,34.938093,-9.856322,Lake Nyasa,Iringa,11,...,user-group,pay annually,soft,good,enough,spring,spring,groundwater,communal standpipe,functional
1,0.0,2013-03-06,Grumeti,1399,GRUMETI,34.698766,-2.147466,Lake Victoria,Mara,20,...,user-group,never pay,soft,good,insufficient,rainwater harvesting,rainwater harvesting,surface,communal standpipe,functional
2,25.0,2013-02-25,Lottery Club,686,World vision,37.460664,-3.821329,Pangani,Manyara,21,...,user-group,pay per bucket,soft,good,enough,dam,dam,surface,communal standpipe multiple,functional
3,0.0,2013-01-28,Unicef,263,UNICEF,38.486161,-11.155298,Ruvuma / Southern Coast,Mtwara,90,...,user-group,never pay,soft,good,dry,machine dbh,borehole,groundwater,communal standpipe multiple,non functional
4,0.0,2011-07-13,Action In A,0,Artisan,31.130847,-1.825359,Lake Victoria,Kagera,18,...,other,never pay,soft,good,seasonal,rainwater harvesting,rainwater harvesting,surface,communal standpipe,functional
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59395,10.0,2013-05-03,Germany Republi,1210,CES,37.169807,-3.253847,Pangani,Kilimanjaro,3,...,user-group,pay per bucket,soft,good,enough,spring,spring,groundwater,communal standpipe,functional
59396,4700.0,2011-05-07,Cefa-njombe,1212,Cefa,35.249991,-9.070629,Rufiji,Iringa,11,...,user-group,pay annually,soft,good,enough,river,river/lake,surface,communal standpipe,functional
59397,0.0,2011-04-11,Unknown,0,Unknown,34.017087,-8.750434,Rufiji,Mbeya,12,...,user-group,pay monthly,fluoride,fluoride,enough,machine dbh,borehole,groundwater,hand pump,functional
59398,0.0,2011-03-08,Malec,0,Musa,35.861315,-6.378573,Rufiji,Dodoma,1,...,user-group,never pay,soft,good,insufficient,shallow well,shallow well,groundwater,hand pump,functional


In [13]:
df_clean = group_rare_categories(df_clean, cols=["district_code", "region_code"], top_n=5, other_label="Others")

Grouped rare categories in: ['region_code', 'district_code']


In [14]:
df_clean = log_transform_skewed_columns(df_clean, exclude_cols=["longitude", "latitude"], skew_threshold=1, inplace=False)

Log transformed: amount_tsh (skew=57.55)
Log transformed: population (skew=12.62)


In [15]:
df_clean = cap_outliers_iqr(df_clean, exclude_cols=["longitude", "latitude"], factor=1.5, inplace=False)

Outliers capped: amount_tsh
Outliers capped: gps_height
Outliers capped: population


In [16]:
save_cleaned_data(df_clean, "../data/df_clean.csv")

Saved cleaned data to: ../data/df_clean.csv
